# Experiment 8: Abstractive Text Summarization (Hindi)

Compare **mT5** and **mBART** on the Hindi split of the XL-Sum dataset using ROUGE-1/2/L and mBERT score.

## 0. Setup & Data Loading

In [1]:
import os, json, tarfile, random
import numpy as np
import torch
from rouge_score import rouge_scorer

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', device)

# Load Hindi XL-Sum from cached tar (or download)
data_dir = 'data'
os.makedirs(data_dir, exist_ok=True)
train_path = os.path.join(data_dir, 'hindi_train.jsonl')
if not os.path.exists(train_path):
    cached = '/Users/himanshu/.cache/huggingface/hub/datasets--csebuetnlp--xlsum/snapshots/30fece425f9a3866e04321773ca7a80056d55ca6/data/hindi_XLSum_v2.0.tar.bz2'
    if os.path.exists(cached):
        tar_path = cached
    else:
        from huggingface_hub import hf_hub_download
        tar_path = hf_hub_download(repo_id='csebuetnlp/xlsum', filename='data/hindi_XLSum_v2.0.tar.bz2', repo_type='dataset')
    with tarfile.open(tar_path) as tar:
        tar.extractall(data_dir)
    print('Extracted data to', data_dir)

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl(os.path.join(data_dir, 'hindi_train.jsonl'))
test_data = load_jsonl(os.path.join(data_dir, 'hindi_test.jsonl'))
print('Train:', len(train_data), 'Test:', len(test_data))

random.seed(42)
eval_samples = random.sample(test_data, 200)
texts = [d['text'] for d in eval_samples]
refs = [d['summary'] for d in eval_samples]
print('Eval samples:', len(texts))


Device: mps


Extracted data to data


Train: 70778 Test: 8847
Eval samples: 200


## 1. mT5 (Multilingual T5)

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

mt5_name = 'Harsit/mt5-small-finetuned-multilingual-xlsum-new'
mt5_tok = AutoTokenizer.from_pretrained(mt5_name)
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(mt5_name).to(device).eval()

mt5_preds = []
for i, t in enumerate(texts):
    inp = mt5_tok(t, return_tensors='pt', truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = mt5_model.generate(**inp, max_length=64, num_beams=4)
    mt5_preds.append(mt5_tok.decode(out[0], skip_special_tokens=True))
    if (i+1) % 50 == 0:
        print(f'mT5: {i+1}/{len(texts)}')

print('Sample prediction:', mt5_preds[0])


/Users/himanshu/Projects/coding/NLP Lab/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 192/192 [00:00<00:00, 26140.37it/s]


[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


mT5: 50/200


mT5: 100/200


mT5: 150/200


mT5: 200/200
Sample prediction: भारतीय तेल मंत्री मुरली देवड़ा ने भारतीय टीवी चैनल से बातचीत के बाद भारतीय तेल मंत्रियों की बैठक में हिस्सा ले रहे थे.


## 2. mBART

In [3]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

mbart_name = 'mskayacioglu/mbart50-xlsum'
mbart_tok = MBart50TokenizerFast.from_pretrained(mbart_name)
mbart_model = MBartForConditionalGeneration.from_pretrained(mbart_name).to(device).eval()

mbart_preds = []
for i, t in enumerate(texts):
    mbart_tok.src_lang = 'hi_IN'
    inp = mbart_tok(t, return_tensors='pt', truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = mbart_model.generate(**inp, forced_bos_token_id=mbart_tok.lang_code_to_id['hi_IN'],
                                  max_length=64, num_beams=4)
    mbart_preds.append(mbart_tok.decode(out[0], skip_special_tokens=True))
    if (i+1) % 50 == 0:
        print(f'mBART: {i+1}/{len(texts)}')

print('Sample prediction:', mbart_preds[0])


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 516/516 [00:00<00:00, 8804.68it/s]

mBART: 50/200


mBART: 100/200


mBART: 150/200


mBART: 200/200
Sample prediction: भारतीय तेल मंत्री मुरली देवड़ा ने कहा है कि ईरान-पाकिस्तान-भारत गैस पाइपलाइन में कुछ दिक्कतें सुलझा ली गई हैं.


## 3. Evaluation: ROUGE & mBERT Score

In [4]:
# ROUGE with whitespace tokenizer for Hindi
class WSTokenizer:
    def tokenize(self, text):
        return text.split()

ws_tok = WSTokenizer()
rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False, tokenizer=ws_tok)

def rouge_avg(preds, refs):
    r1 = r2 = rl = 0.0
    for p, r in zip(preds, refs):
        s = rouge.score(r, p)
        r1 += s['rouge1'].fmeasure
        r2 += s['rouge2'].fmeasure
        rl += s['rougeL'].fmeasure
    n = len(preds)
    return r1/n, r2/n, rl/n

# mBERT score: cosine similarity of mean-pooled multilingual BERT embeddings
from transformers import AutoTokenizer as MBertTok, AutoModel as MBertModel
mbert_tok = MBertTok.from_pretrained('bert-base-multilingual-cased')
mbert_model = MBertModel.from_pretrained('bert-base-multilingual-cased').to(device).eval()

def mbert_embed(text):
    inp = mbert_tok(text, return_tensors='pt', truncation=True, max_length=128).to(device)
    with torch.no_grad():
        out = mbert_model(**inp)
    return out.last_hidden_state.mean(dim=1)

def mbert_score(preds, refs):
    sims = []
    for p, r in zip(preds, refs):
        a = mbert_embed(p)
        b = mbert_embed(r)
        sims.append(torch.nn.functional.cosine_similarity(a, b).item())
    return float(np.mean(sims))

print('Computing mT5 metrics...')
mt5_r1, mt5_r2, mt5_rl = rouge_avg(mt5_preds, refs)
mt5_mb = mbert_score(mt5_preds, refs)
print('Computing mBART metrics...')
mbart_r1, mbart_r2, mbart_rl = rouge_avg(mbart_preds, refs)
mbart_mb = mbert_score(mbart_preds, refs)

import pandas as pd
df = pd.DataFrame({
    'Model': ['mT5', 'mBART'],
    'ROUGE-1': [round(mt5_r1,4), round(mbart_r1,4)],
    'ROUGE-2': [round(mt5_r2,4), round(mbart_r2,4)],
    'ROUGE-L': [round(mt5_rl,4), round(mbart_rl,4)],
    'mBERT': [round(mt5_mb,4), round(mbart_mb,4)],
})
print(df.to_string(index=False))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7319.32it/s]


[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing mT5 metrics...


Computing mBART metrics...


Model  ROUGE-1  ROUGE-2  ROUGE-L  mBERT
  mT5   0.2357   0.0663   0.1963 0.7726
mBART   0.3350   0.1436   0.2831 0.8202


## 4. Sample Outputs

In [5]:
for i in range(3):
    print(f"--- Sample {i+1} ---")
    print("TEXT:", texts[i][:200].replace(chr(10),' '))
    print("REF :", refs[i])
    print("mT5 :", mt5_preds[i])
    print("mBART:", mbart_preds[i])
    print()


--- Sample 1 ---
TEXT: रविवार को जेद्दा में ईरानी तेल मंत्री ग़ुलाम हुसैन नज़ारी से बातचीत के बाद भारतीय तेल मंत्री मुरली देवड़ा ने एक भारतीय टीवी चैनल से ये बात कही. मुरली देवड़ा सऊदी शहर जेद्दा में तेल मंत्रियों की बैठक म
REF : भारत का कहना है कि वो जल्द ही ईरान और पाकिस्तान के साथ गैस पाइपलाइन समझौते पर हस्ताक्षर कर देगा.
mT5 : भारतीय तेल मंत्री मुरली देवड़ा ने भारतीय टीवी चैनल से बातचीत के बाद भारतीय तेल मंत्रियों की बैठक में हिस्सा ले रहे थे.
mBART: भारतीय तेल मंत्री मुरली देवड़ा ने कहा है कि ईरान-पाकिस्तान-भारत गैस पाइपलाइन में कुछ दिक्कतें सुलझा ली गई हैं.

--- Sample 2 ---
TEXT: इसके पहले कोफ़ी अन्नान सीरिया गए थे. उन्होंने कहा कि हिज़्बुल्ला को हथियारों की तस्करी रोकने पर उन्हें सीरिया की ओर से सकारात्मक संकेत मिले हैं. तेहरान से बीबीसी संवाददाता का कहना है कि कोफ़ी अन्नान ई
REF : संयुक्त राष्ट्र महासचिव कोफ़ी अन्नान ईरानी नेताओं से बातचीत के लिए तेहरान पहुँच गए हैं. बातचीत का मक़सद लेबनान में युद्धविराम को मज़बूती प्रदान करना है.
mT5 : ईरान के राष्ट्रपति कोफ़ी अन्नान सीरिया म